Organize Slices Patient-wise

In [2]:
import os
import shutil
from tqdm import tqdm

master_folder = r"C:\Users\Rusab\Downloads\New folder (12)\thirdMethod_MAIN_AP\masks"

def extract_identifier(filename):
    return filename.split('_')[1]

def create_and_copy(src_path, dest_folder):
    if not os.path.exists(dest_folder):
        os.makedirs(dest_folder)
    shutil.copy(src_path, dest_folder)

def create_and_move(src_path, dest_folder):
    if not os.path.exists(dest_folder):
        os.makedirs(dest_folder)
    shutil.move(src_path, dest_folder)

for filename in tqdm(os.listdir(master_folder)):
    if filename.endswith(".png"):
        identifier = extract_identifier(filename)
        dest_folder = os.path.join(master_folder, identifier)
        src_path = os.path.join(master_folder, filename)
        #create_and_copy(src_path, dest_folder)
        create_and_move(src_path, dest_folder)

print("Files have been organized!")


100%|██████████| 5747/5747 [00:03<00:00, 1842.03it/s]

Files have been organized!


Stack N slices

In [ ]:
import os
import numpy as np
from PIL import Image
import tifffile
import glob
from pathlib import Path
import shutil

def create_tiff_stacks(ct_base_dir, mask_base_dir, output_base_dir):
    """
    Create 5-channel TIFF stacks from CT slices with corresponding tumor masks.
    
    Args:
        ct_base_dir: Directory containing patient folders with CT slices
        mask_base_dir: Directory containing patient folders with tumor masks
        output_base_dir: Directory to save the output TIFF stacks and masks
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_base_dir, exist_ok=True)
    
    # Get list of patient directories
    patient_dirs = [d for d in os.listdir(ct_base_dir) if os.path.isdir(os.path.join(ct_base_dir, d))]
    
    for patient_id in patient_dirs:
        print(f"Processing patient: {patient_id}")
        
        # Path to patient's CT slices and masks
        ct_dir = os.path.join(ct_base_dir, patient_id)
        mask_dir = os.path.join(mask_base_dir, patient_id)
        
        # Create output directory for this patient
        patient_output_dir = os.path.join(output_base_dir, patient_id)
        os.makedirs(patient_output_dir, exist_ok=True)
        
        # Get sorted list of CT slice filenames and mask filenames
        ct_files = sorted(glob.glob(os.path.join(ct_dir, "*.png")))  # Adjust extension if needed
        mask_files = sorted(glob.glob(os.path.join(mask_dir, "*.png")))  # Adjust extension if needed
        
        if not ct_files:
            print(f"No CT files found for patient {patient_id}")
            continue
            
        if not mask_files:
            print(f"No mask files found for patient {patient_id}")
            continue
        
        # Read first CT slice to get dimensions
        first_slice = np.array(Image.open(ct_files[0]))
        slice_height, slice_width = first_slice.shape
        
        # Create a blank slice for padding
        blank_slice = np.zeros((slice_height, slice_width), dtype=np.uint8)
        
        # Process all mask files
        for mask_idx, mask_file in enumerate(mask_files):
            mask_name = os.path.basename(mask_file)
            mask_basename = os.path.splitext(mask_name)[0]
            
            # Find the corresponding CT slice index
            try:
                ct_idx = [os.path.basename(f) for f in ct_files].index(mask_name)
            except ValueError:
                print(f"Warning: No matching CT file for mask {mask_name}. Skipping...")
                continue
            
            # Create a stack of 5 slices centered at the current slice
            stack = []
            
            # Add slices to the stack (with padding if needed)
            for i in range(ct_idx - N_slices//2, ct_idx + N_slices//2 + 1):
                if i < 0:
                    # Pad with blank slice if we're near the beginning
                    stack.append(blank_slice)
                elif i >= len(ct_files):
                    # Pad with blank slice if we're near the end
                    stack.append(blank_slice)
                else:
                    # Add the actual CT slice
                    ct_slice = np.array(Image.open(ct_files[i]))
                    stack.append(ct_slice)
            
            # Convert list of slices to 3D numpy array (C, H, W)
            stack_array = np.array(stack, dtype=np.uint8)
            
            # Save as 5-channel TIFF
            tiff_output_path = os.path.join(patient_output_dir, f"{mask_basename}.tif")
            tifffile.imwrite(tiff_output_path, stack_array)
            
            
        print(f"Completed processing patient {patient_id}")

if __name__ == "__main__":
    # Update these paths for your specific directory structure
    ct_base_directory = r"C:\Users\Rusab\Downloads\New folder (12)\thirdMethod_MAIN_AP\images"  # Base directory containing patient folders with CT slices
    mask_base_directory = r"C:\Users\Rusab\Downloads\New folder (12)\thirdMethod_MAIN_AP\masks"  # Base directory containing patient folders with tumor masks
    output_directory = r"C:\Users\Rusab\Downloads\New folder (12)\thirdMethod_MAIN_AP\output"  # Directory to save the processed data
    N_slices = 5 
    
    create_tiff_stacks(ct_base_directory, mask_base_directory, output_directory)
    print("Processing complete!")

Processing patient: 1
Completed processing patient 1
Processing patient: 100
Completed processing patient 100
Processing patient: 101
Completed processing patient 101
Processing patient: 102
Completed processing patient 102
Processing patient: 103
Completed processing patient 103
Processing patient: 104
Completed processing patient 104
Processing patient: 105
Completed processing patient 105
Processing patient: 11
Completed processing patient 11
Processing patient: 12
Completed processing patient 12
Processing patient: 13
Completed processing patient 13
Processing patient: 18
Completed processing patient 18
Processing patient: 2
Completed processing patient 2
Processing patient: 20
Completed processing patient 20
Processing patient: 22
Completed processing patient 22
Processing patient: 23
Completed processing patient 23
Processing patient: 25
Completed processing patient 25
Processing patient: 26
Completed processing patient 26
Processing patient: 28
Completed processing patient 28
Pr

Check the TIFF Stacks

In [22]:
#load and show the tiff file
import tifffile as tiff

import matplotlib.pyplot as plt

import numpy as np


# Load the TIFF file
tiff_file_path = r"e:\Code Works\testing-2d-segmentation\2.5D_added\Data\Train\fold_1\images\HCC_8_4389.tif"
print(os.path.splitext(tiff_file_path)[0])  # Check the file extension

tiff_data = tiff.imread(tiff_file_path)
tiff_data = tiff_data.astype(np.uint8)
tiff_data = np.transpose(tiff_data, (1, 2, 0))

print(tiff_data.shape)  # Check the shape of the loaded data

print(tiff_data.dtype)  # Check the data type of the loaded data

# Display the first slice of the TIFF stack

plt.imshow(tiff_data[4], cmap='gray')
plt.axis('off')
plt.show()


e:\Code Works\testing-2d-segmentation\2.5D_added\Data\Train\fold_1\images\HCC_8_4389
(512, 512, 5)
uint8


K-Fold Patient Wise Split

In [23]:
import os
import shutil
import numpy as np
from tqdm import tqdm

#Declare seed for reproducibility
np.random.seed(0)

def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)


def copy_subject_data(subject, source_root, dest_root, data_type, category):
    src_dir = os.path.join(source_root, subject)
    dest_dir = os.path.join(dest_root, "Data", data_type, fold_dir, category)

    ensure_dir(dest_dir)

    for file_name in os.listdir(src_dir):
        src_file = os.path.join(src_dir, file_name)
        dest_file = os.path.join(dest_dir, file_name)

        shutil.copy2(src_file, dest_file)

def subjectwise_cross_validation(images_folder, masks_folder, dest_root, n_folds, val_percent):
    # List all the subject folders
    subjects = [d for d in os.listdir(images_folder) if os.path.isdir(os.path.join(images_folder, d))]
    np.random.shuffle(subjects)

    # Split the subjects into n-folds
    split_size = len(subjects) // n_folds
    remainder = len(subjects) % n_folds
    splits = []
    
    start = 0
    for i in range(n_folds):
        end = start + split_size + (1 if i < remainder else 0)
        splits.append(subjects[start:end])
        start = end

    # Create cross-validation folds
    for i, test_subjects in enumerate(splits):
        global fold_dir  # Make fold_dir global so it can be accessed in copy_subject_data
        fold_dir = f"fold_{i+1}"
        train_subjects = [subj for subj in subjects if subj not in test_subjects]
        num_val = int(len(train_subjects) * val_percent / 100)
        val_subjects = train_subjects[:num_val]
        train_subjects = train_subjects[num_val:]

        # Copy test, train, val data
        for test_subj in tqdm(test_subjects, desc=f'Fold_{i+1} Test: ', leave= False):
            copy_subject_data(test_subj, images_folder, dest_root, "Test", "images")
            copy_subject_data(test_subj, masks_folder, dest_root, "Test", "masks")

        for train_subj in tqdm(train_subjects, desc=f'Fold_{i+1} Train: ', leave= False):
            copy_subject_data(train_subj, images_folder, dest_root, "Train", "images")
            copy_subject_data(train_subj, masks_folder, dest_root, "Train", "masks")

        for val_subj in tqdm(val_subjects, desc=f'Fold_{i+1} Val: ', leave= False):
            copy_subject_data(val_subj, images_folder, dest_root, "Val", "images")
            copy_subject_data(val_subj, masks_folder, dest_root, "Val", "masks")

In [ ]:
images_folder = r'C:\Users\Rusab\Downloads\New folder (12)\thirdMethod_MAIN_AP\output'
masks_folder = r'C:\Users\Rusab\Downloads\New folder (12)\thirdMethod_MAIN_AP\masks'
dest_root = r'E:\Code Works\testing-2d-segmentation\2.5D_added'
n_folds = 5
val_percent = 20

subjectwise_cross_validation(images_folder, masks_folder, dest_root, n_folds, val_percent)

2.5D Augmentation (Alpha) : Use at your own risk

In [ ]:
import os
import albumentations as A
import cv2
import numpy as np
from tqdm import tqdm
import random
import tifffile
from PIL import Image

random.seed(0)

def augmentImage(img_stack, is_mask=False):
    """
    Apply augmentations to a stack of images (TIFF with multiple channels)
    or to a single mask.
    
    Args:
        img_stack: 3D array (C, H, W) for TIFF stack or 2D array (H, W) for mask
        is_mask: Boolean indicating if the input is a mask
    
    Returns:
        List of augmented image stacks or masks
    """
    transform_list = [
        A.Rotate(limit=[90, 90], p=1, border_mode=cv2.BORDER_CONSTANT),
        A.Rotate(limit=[-90, -90], p=1, border_mode=cv2.BORDER_CONSTANT),
        A.Rotate(limit=[-30, -30], p=1, border_mode=cv2.BORDER_CONSTANT),
        A.Rotate(limit=[30, 30], p=1, border_mode=cv2.BORDER_CONSTANT),
        A.HorizontalFlip(p=1),
    ]
    
    augmented_results = []
    
    if is_mask:
        # For masks (single 2D image)
        for i in range(len(transform_list)):
            transform = A.Compose([transform_list[i]])
            transformed = transform(image=img_stack)
            augmented_results.append(transformed['image'])
    else:
        # For TIFF stacks (multiple channels)
        for i in range(len(transform_list)):
            transform = A.Compose([transform_list[i]])
            
            # Apply the same transformation to each channel
            transformed_stack = []
            for channel in range(img_stack.shape[0]):
                # Extract single channel
                channel_img = img_stack[channel, :, :]
                
                # Apply the transformation
                transformed = transform(image=channel_img)
                
                # Add the transformed channel
                transformed_stack.append(transformed['image'])
            
            # Stack the transformed channels
            transformed_stack = np.array(transformed_stack)
            augmented_results.append(transformed_stack)
    
    return augmented_results

def save_tiff_stack(img_stack, dir, filename):
    """
    Save 5-channel TIFF stack.
    
    Args:
        img_stack: 3D array (C, H, W)
        dir: Output directory
        filename: Base filename without extension
    """
    img_filename = os.path.join(dir, filename + '.tif')
    tifffile.imwrite(img_filename, img_stack.astype(np.uint8))

def save_mask(img, dir, filename):
    """
    Save binary mask as PNG.
    """
    img_dir = dir
    img_filename = os.path.join(img_dir, filename + '.png')
    _, img = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    cv2.imwrite(img_filename, img)

def save_mask_multiclass(img, dir, filename):
    """
    Save multiclass mask as PNG with palette.
    """
    img_dir = dir
    img_filename = os.path.join(img_dir, filename + '.png')
    processed_mask = Image.fromarray(img)
    processed_mask = processed_mask.convert('P')
    processed_mask.putpalette(palette)
    processed_mask.save(img_filename)

def main():
    main_dir = r'E:\Code Works\CCM-Project\Annotation\test\Reannoation\2nd_revision\Data\Train'
    multiclass = False
    palette = [0, 0, 0, 255, 0, 0, 0, 0, 255]
    n = 5
    
    for fold in range(1, n+1):
        # Define directories
        tiff_dir = os.path.join(main_dir, f'fold_{fold}', 'tiff_stacks')
        mask_dir = os.path.join(main_dir, f'fold_{fold}', 'masks')
        
        # Create output directories if they don't exist
        os.makedirs(tiff_dir, exist_ok=True)
        os.makedirs(mask_dir, exist_ok=True)
        
        # Get all TIFF files and masks
        tiff_files = [os.path.join(tiff_dir, x) for x in os.listdir(tiff_dir) if x.endswith('.tif')]
        mask_files = [os.path.join(mask_dir, x) for x in os.listdir(mask_dir) if x.endswith('.png')]
        
        # Match TIFF files to mask files
        for tiff_path in tqdm(tiff_files, desc=f'Fold_{fold}: '):
            base_name = os.path.basename(tiff_path).replace('.tif', '')
            mask_path = os.path.join(mask_dir, base_name + '.png')
            
            if not os.path.exists(mask_path):
                print(f"Warning: No matching mask for {tiff_path}")
                continue
            
            # Load TIFF stack and mask
            tiff_stack = tifffile.imread(tiff_path)
            
            if multiclass:
                mask = np.array(Image.open(mask_path))
            else:
                mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
            
            # Augment TIFF stack and mask
            aug_tiff_stacks = augmentImage(tiff_stack, is_mask=False)
            aug_masks = augmentImage(mask, is_mask=True)
            
            # Save augmented results
            for i in range(len(aug_tiff_stacks)):
                aug_name = base_name + 'a' + str(i)
                
                # Save the augmented TIFF stack
                save_tiff_stack(aug_tiff_stacks[i], tiff_dir, aug_name)
                
                # Save the augmented mask
                if multiclass:
                    save_mask_multiclass(aug_masks[i], mask_dir, aug_name)
                else:
                    save_mask(aug_masks[i], mask_dir, aug_name)

if __name__ == "__main__":
    main()